In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr, col
spark = SparkSession.builder.appName("class-3-1").getOrCreate()

spark


25/07/11 22:49:09 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
events = (spark.read.option("header", "true")
          .csv("/home/iceberg/data/events.csv")
          .withColumn("event_date", expr("DATE_TRUNC('day', event_time)"))
         )
devices = spark.read.option("header","true").csv("/home/iceberg/data/devices.csv")

df = events.join(devices,on="device_id",how="left")
df = df.withColumnsRenamed({'browser_type': 'browser_family', 'os_type': 'os_family'})

df.show(10)

+---------+-----------+--------+--------------------+---+--------------------+-------------------+--------------+---------+-----------+
|device_id|    user_id|referrer|                host|url|          event_time|         event_date|browser_family|os_family|device_type|
+---------+-----------+--------+--------------------+---+--------------------+-------------------+--------------+---------+-----------+
|532630305| 1037710827|    NULL| www.zachwilson.tech|  /|2021-03-08 17:27:...|2021-03-08 00:00:00|         Other|    Other|      Other|
|532630305|  925588856|    NULL|    www.eczachly.com|  /|2021-05-10 11:26:...|2021-05-10 00:00:00|         Other|    Other|      Other|
|532630305|-1180485268|    NULL|admin.zachwilson....|  /|2021-02-17 16:19:...|2021-02-17 00:00:00|         Other|    Other|      Other|
|532630305|-1044833855|    NULL| www.zachwilson.tech|  /|2021-09-24 15:53:...|2021-09-24 00:00:00|         Other|    Other|      Other|
|532630305|  747494706|    NULL| www.zachwilson.

In [12]:
df.count()

404814

In [13]:
df.collect()

[Row(device_id='532630305', user_id='1037710827', referrer=None, host='www.zachwilson.tech', url='/', event_time='2021-03-08 17:27:24.241000', event_date=datetime.datetime(2021, 3, 8, 0, 0), browser_family='Other', os_family='Other', device_type='Other'),
 Row(device_id='532630305', user_id='925588856', referrer=None, host='www.eczachly.com', url='/', event_time='2021-05-10 11:26:21.247000', event_date=datetime.datetime(2021, 5, 10, 0, 0), browser_family='Other', os_family='Other', device_type='Other'),
 Row(device_id='532630305', user_id='-1180485268', referrer=None, host='admin.zachwilson.tech', url='/', event_time='2021-02-17 16:19:30.738000', event_date=datetime.datetime(2021, 2, 17, 0, 0), browser_family='Other', os_family='Other', device_type='Other'),
 Row(device_id='532630305', user_id='-1044833855', referrer=None, host='www.zachwilson.tech', url='/', event_time='2021-09-24 15:53:14.466000', event_date=datetime.datetime(2021, 9, 24, 0, 0), browser_family='Other', os_family='Oth

In [16]:
# forcing java.lang.OutOfMemoryError: Java heap space

from pyspark.sql.functions import lit
df.join(df, lit(1) == lit(1)).collect()

# collect tries to get all data into driver memory, and for this case it ran into OOO. Check example below using take(5)

25/07/11 22:46:38 WARN Column: Constructing trivially true equals predicate, '1 = 1'. Perhaps you need to use aliases.
25/07/11 22:46:44 ERROR Executor: Exception in task 0.0 in stage 51.0 (TID 95)6]
java.lang.OutOfMemoryError: Java heap space
	at java.base/java.nio.HeapByteBuffer.<init>(HeapByteBuffer.java:64)
	at java.base/java.nio.ByteBuffer.allocate(ByteBuffer.java:363)
	at org.apache.spark.sql.execution.SparkPlan.$anonfun$getByteArrayRdd$2(SparkPlan.scala:368)
	at org.apache.spark.sql.execution.SparkPlan.$anonfun$getByteArrayRdd$2$adapted(SparkPlan.scala:368)
	at org.apache.spark.sql.execution.SparkPlan$$Lambda$2985/0x0000740fe8f52768.apply(Unknown Source)
	at org.apache.spark.util.io.ChunkedByteBufferOutputStream.allocateNewChunkIfNeeded(ChunkedByteBufferOutputStream.scala:87)
	at org.apache.spark.util.io.ChunkedByteBufferOutputStream.write(ChunkedByteBufferOutputStream.scala:75)
	at net.jpountz.lz4.LZ4BlockOutputStream.flushBufferedData(LZ4BlockOutputStream.java:225)
	at net.jpo

ConnectionRefusedError: [Errno 111] Connection refused

In [5]:
# after kill the kernel by OOO (out of memory), one needs to restart it before run this cell

from pyspark.sql.functions import lit

df.join(df, lit(1) == lit(1)).take(5)

# take 5 avoid going OOO

25/07/11 22:50:31 WARN Column: Constructing trivially true equals predicate, '1 = 1'. Perhaps you need to use aliases.


[Row(device_id='532630305', user_id='1037710827', referrer=None, host='www.zachwilson.tech', url='/', event_time='2021-03-08 17:27:24.241000', event_date=datetime.datetime(2021, 3, 8, 0, 0), browser_family='Other', os_family='Other', device_type='Other', device_id='532630305', user_id='1037710827', referrer=None, host='www.zachwilson.tech', url='/', event_time='2021-03-08 17:27:24.241000', event_date=datetime.datetime(2021, 3, 8, 0, 0), browser_family='Other', os_family='Other', device_type='Other'),
 Row(device_id='532630305', user_id='1037710827', referrer=None, host='www.zachwilson.tech', url='/', event_time='2021-03-08 17:27:24.241000', event_date=datetime.datetime(2021, 3, 8, 0, 0), browser_family='Other', os_family='Other', device_type='Other', device_id='532630305', user_id='925588856', referrer=None, host='www.eczachly.com', url='/', event_time='2021-05-10 11:26:21.247000', event_date=datetime.datetime(2021, 5, 10, 0, 0), browser_family='Other', os_family='Other', device_type='

In [8]:
sorted = events.repartition(10, col("event_date"))\
    .sortWithinPartitions(col("event_date"), col("host"))\
    .withColumn("event_time", col("event_time").cast("timestamp")) 

sortedTwo = events.repartition(10, col("event_date"))\
    .sort(col("event_date"), col("host"))\
    .withColumn("event_time", col("event_time").cast("timestamp")) 

sorted.show(5)
sortedTwo.show(5)


+-----------+----------+--------+--------------------+--------------------+--------------------+-------------------+
|    user_id| device_id|referrer|                host|                 url|          event_time|         event_date|
+-----------+----------+--------+--------------------+--------------------+--------------------+-------------------+
| 1129583063| 532630305|    NULL|admin.zachwilson....|                   /|2021-01-07 09:21:...|2021-01-07 00:00:00|
| -648945006|1088283544|    NULL|    www.eczachly.com|                   /|2021-01-07 02:58:...|2021-01-07 00:00:00|
|-1871780024|-158310583|    NULL|    www.eczachly.com|                   /|2021-01-07 04:17:...|2021-01-07 00:00:00|
|  203689086|1088283544|    NULL|    www.eczachly.com|/blog/what-exactl...|2021-01-07 10:03:...|2021-01-07 00:00:00|
|-1180485268| 532630305|    NULL|    www.eczachly.com|                   /|2021-01-07 18:45:...|2021-01-07 00:00:00|
+-----------+----------+--------+--------------------+----------

[Stage 9:=============================>                             (2 + 2) / 4]

+----------+----------+--------+--------------------+-------------+--------------------+-------------------+
|   user_id| device_id|referrer|                host|          url|          event_time|         event_date|
+----------+----------+--------+--------------------+-------------+--------------------+-------------------+
|1272828233|-643696601|    NULL|admin.zachwilson....|            /|2021-01-02 13:53:...|2021-01-02 00:00:00|
| 747494706| 532630305|    NULL|admin.zachwilson....|            /|2021-01-02 19:36:...|2021-01-02 00:00:00|
|2110046626| 898871897|    NULL|admin.zachwilson....|/wp-login.php|2021-01-02 19:57:...|2021-01-02 00:00:00|
|1272828233|-643696601|    NULL|admin.zachwilson....|            /|2021-01-02 21:05:...|2021-01-02 00:00:00|
|1272828233|-643696601|    NULL|admin.zachwilson....|            /|2021-01-02 21:37:...|2021-01-02 00:00:00|
+----------+----------+--------+--------------------+-------------+--------------------+-------------------+
only showing top 5 

In [9]:
sorted.explain()

sortedTwo.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [user_id#17, device_id#18, referrer#19, host#20, url#21, cast(event_time#22 as timestamp) AS event_time#208, event_date#29]
   +- Sort [event_date#29 ASC NULLS FIRST, host#20 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(event_date#29, 10), REPARTITION_BY_NUM, [plan_id=397]
         +- Project [user_id#17, device_id#18, referrer#19, host#20, url#21, event_time#22, date_trunc(day, cast(event_time#22 as timestamp), Some(Etc/UTC)) AS event_date#29]
            +- FileScan csv [user_id#17,device_id#18,referrer#19,host#20,url#21,event_time#22] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/iceberg/data/events.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<user_id:string,device_id:string,referrer:string,host:string,url:string,event_time:string>


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [user_id#17, device_id#18, referr

In [10]:
# now without the repartitions

sorted = events \
    .sortWithinPartitions(col("event_date"), col("host"))\
    .withColumn("event_time", col("event_time").cast("timestamp")) 

sortedTwo = events \
    .sort(col("event_date"), col("host"))\
    .withColumn("event_time", col("event_time").cast("timestamp")) 

sorted.explain()

sortedTwo.explain()

# to do a global sort (sort command), the data necessarily needs to go to a final single executor


# .sortWithinPartitions() sorts within partitions, whereas .sort() is a global sort, which is very slow

# Note - exchange is synonymous with Shuffle

== Physical Plan ==
*(1) Project [user_id#17, device_id#18, referrer#19, host#20, url#21, cast(event_time#22 as timestamp) AS event_time#294, event_date#29]
+- *(1) Sort [event_date#29 ASC NULLS FIRST, host#20 ASC NULLS FIRST], false, 0
   +- *(1) Project [user_id#17, device_id#18, referrer#19, host#20, url#21, event_time#22, date_trunc(day, cast(event_time#22 as timestamp), Some(Etc/UTC)) AS event_date#29]
      +- FileScan csv [user_id#17,device_id#18,referrer#19,host#20,url#21,event_time#22] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/iceberg/data/events.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<user_id:string,device_id:string,referrer:string,host:string,url:string,event_time:string>


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [user_id#17, device_id#18, referrer#19, host#20, url#21, cast(event_time#22 as timestamp) AS event_time#302, event_date#29]
   +- Sort [event_date#29 ASC NULLS F

In [11]:
# start sql preparation

In [15]:
%%sql 
show databases

namespace


In [16]:
%%sql

CREATE DATABASE IF NOT EXISTS bootcamp

++
||
++
++

In [17]:
%%sql 
show databases

namespace
bootcamp


In [20]:
%%sql

DROP TABLE IF EXISTS bootcamp.events;

++
||
++
++

In [21]:
%%sql

CREATE TABLE IF NOT EXISTS bootcamp.events (
    url STRING,
    referrer STRING,
    browser_family STRING,
    os_family STRING,
    device_family STRING,
    host STRING,
    event_time TIMESTAMP,
    event_date DATE
)
USING iceberg
PARTITIONED BY (years(event_date));


++
||
++
++

In [22]:
%%sql

DROP TABLE IF EXISTS bootcamp.events_sorted;

++
||
++
++

In [23]:
%%sql

CREATE TABLE IF NOT EXISTS bootcamp.events_sorted (
    url STRING,
    referrer STRING,
    browser_family STRING,
    os_family STRING,
    device_family STRING,
    host STRING,
    event_time TIMESTAMP,
    event_date DATE
)
USING iceberg
PARTITIONED BY (years(event_date));

++
||
++
++

In [25]:
%%sql
DROP TABLE IF EXISTS bootcamp.events_unsorted;

++
||
++
++

In [26]:
%%sql


CREATE TABLE IF NOT EXISTS bootcamp.events_unsorted (
    url STRING,
    referrer STRING,
    browser_family STRING,
    os_family STRING,
    device_family STRING,
    host STRING,
    event_time TIMESTAMP,
    event_date DATE
)
USING iceberg
PARTITIONED BY (year(event_date));

++
||
++
++

In [27]:

start_df = df.repartition(4, col("event_date")).withColumn("event_time", col("event_time").cast("timestamp")) \
    
first_sort_df = start_df.sortWithinPartitions(col("event_date"), col("host"))

start_df.write.mode("overwrite").saveAsTable("bootcamp.events_unsorted")
first_sort_df.write.mode("overwrite").saveAsTable("bootcamp.events_sorted")

In [28]:
%%sql

SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'sorted' 
FROM demo.bootcamp.events_sorted.files

UNION ALL
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'unsorted' 
FROM demo.bootcamp.events_unsorted.files

size,num_files,sorted
5444946,4,sorted
5556664,4,unsorted


In [30]:
%%sql

SELECT * FROM demo.bootcamp.events_sorted.files

content,file_path,file_format,spec_id,partition,record_count,file_size_in_bytes,column_sizes,value_counts,null_value_counts,nan_value_counts,lower_bounds,upper_bounds,key_metadata,split_offsets,equality_ids,sort_order_id,referenced_data_file,content_offset,content_size_in_bytes,readable_metrics
0,s3://warehouse/bootcamp/events_sorted/data/00000-39-1afab447-bee4-421e-897d-0bbc256b28ff-0-00001.parquet,PARQUET,1,Row(event_date_year=None),89391,1104739,"{1: 111534, 2: 68783, 3: 48260, 4: 25739, 6: 2692, 7: 390807, 8: 2293, 9: 103875, 10: 315050, 11: 30602}","{1: 89391, 2: 89391, 3: 89391, 4: 89391, 6: 89391, 7: 89391, 8: 89391, 9: 89391, 10: 89391, 11: 89391}","{1: 0, 2: 46359, 3: 0, 4: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 1, 11: 0}",{},"{1: bytearray(b'/'), 2: bytearray(b'52.20.78.240'), 3: bytearray(b'%E3%82%A6%E3%82%'), 4: bytearray(b'Android'), 6: bytearray(b'aashish.techcrea'), 7: bytearray(b' \xba\xe7\xb8\xa8\xb8\x05\x00'), 8: bytearray(b'\x00\xa0&\xb4\xa8\xb8\x05\x00'), 9: bytearray(b'-100210680'), 10: bytearray(b'-1000095488'), 11: bytearray(b'17MB150WB')}","{1: bytearray(b'/zzageqnf.php?Fp'), 2: bytearray(b'zachwilson.tech'), 3: bytearray(b'webprosbot'), 4: bytearray(b'iOS'), 6: bytearray(b'zachwilson.techd'), 7: bytearray(b'\xe8\xb0\x1b\x8ec\x03\x06\x00'), 8: bytearray(b'\x00\xe0dqO\x03\x06\x00'), 9: bytearray(b'999535123'), 10: bytearray(b'999884938'), 11: bytearray(b'vivo $2')}",None,[4],None,0,None,None,None,"Row(browser_family=Row(column_size=48260, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound='%E3%82%A6%E3%82%', upper_bound='webprosbot'), device_id=Row(column_size=103875, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound='-100210680', upper_bound='999535123'), device_type=Row(column_size=30602, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound='17MB150WB', upper_bound='vivo $2'), event_date=Row(column_size=2293, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound=datetime.datetime(2021, 1, 12, 0, 0), upper_bound=datetime.datetime(2023, 8, 20, 0, 0)), event_time=Row(column_size=390807, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound=datetime.datetime(2021, 1, 12, 0, 1, 19, 764000), upper_bound=datetime.datetime(2023, 8, 20, 23, 59, 41, 89000)), host=Row(column_size=2692, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound='aashish.techcrea', upper_bound='zachwilson.techd'), os_family=Row(column_size=25739, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound='Android', upper_bound='iOS'), referrer=Row(column_size=68783, value_count=89391, null_value_count=46359, nan_value_count=None, lower_bound='52.20.78.240', upper_bound='zachwilson.tech'), url=Row(column_size=111534, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound='/', upper_bound='/zzageqnf.php?Fp'), user_id=Row(column_size=315050, value_count=89391, null_value_count=1, nan_value_count=None, lower_bound='-1000095488', upper_bound='999884938'))"
0,s3://warehouse/bootcamp/events_sorted/data/00001-40-1afab447-bee4-421e-897d-0bbc256b28ff-0-00001.parquet,PARQUET,1,Row(event_date_year=None),99232,1246012,"{1: 145867, 2: 73774, 3: 48052, 4: 34585, 6: 3312, 7: 435876, 8: 2373, 9: 117107, 10: 344946, 11: 34972}","{1: 99232, 2: 99232, 3: 99232, 4: 99232, 6: 99232, 7: 99232, 8: 99232, 9: 99232, 10: 99232, 11: 99232}","{1: 0, 2: 49299, 3: 0, 4: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 58, 11: 0}",{},"{1: bytearray(b'""/?""""<?=print(93'), 2: bytearray(b'""https://www.goo'), 3: bytearray(b') Bot'), 4: bytearray(b'Android'), 6: bytearray(b'abhishekanand.te'), 7: bytearray(b'(\x83\xb2EX\xb8\x05\x00'), 8: bytearray(b'\x00 \xc9<X\xb8\x05\x00'), 9: bytearray(b'-100210680'), 10: bytearray(b'-1000370060'), 11: bytearray(b'13 Pro Max')}","{1: bytearray(b'/zz.php'), 2: bytearray(b'zachwilson.tech'), 3: bytearray(b'webprosbot'), 4: bytearray(b'iOS'), 6: bytearray(b'zsavi524.techcrf'), 7: bytearray(b'\x88\xb8\x07P;\x03\

In [32]:
%%sql

SELECT * FROM demo.bootcamp.events_unsorted.files

content,file_path,file_format,spec_id,partition,record_count,file_size_in_bytes,column_sizes,value_counts,null_value_counts,nan_value_counts,lower_bounds,upper_bounds,key_metadata,split_offsets,equality_ids,sort_order_id,referenced_data_file,content_offset,content_size_in_bytes,readable_metrics
0,s3://warehouse/bootcamp/events_unsorted/data/00000-30-6494c81b-af5b-4cb8-9814-851149849b65-0-00001.parquet,PARQUET,1,Row(event_date_year=None),89391,1124080,"{1: 113484, 2: 69624, 3: 50044, 4: 26501, 6: 17186, 7: 381905, 8: 4742, 9: 106306, 10: 317883, 11: 31286}","{1: 89391, 2: 89391, 3: 89391, 4: 89391, 6: 89391, 7: 89391, 8: 89391, 9: 89391, 10: 89391, 11: 89391}","{1: 0, 2: 46359, 3: 0, 4: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 1, 11: 0}",{},"{1: bytearray(b'/'), 2: bytearray(b'52.20.78.240'), 3: bytearray(b'%E3%82%A6%E3%82%'), 4: bytearray(b'Android'), 6: bytearray(b'aashish.techcrea'), 7: bytearray(b' \xba\xe7\xb8\xa8\xb8\x05\x00'), 8: bytearray(b'\x00\xa0&\xb4\xa8\xb8\x05\x00'), 9: bytearray(b'-100210680'), 10: bytearray(b'-1000095488'), 11: bytearray(b'17MB150WB')}","{1: bytearray(b'/zzageqnf.php?Fp'), 2: bytearray(b'zachwilson.tech'), 3: bytearray(b'webprosbot'), 4: bytearray(b'iOS'), 6: bytearray(b'zachwilson.techd'), 7: bytearray(b'\xe8\xb0\x1b\x8ec\x03\x06\x00'), 8: bytearray(b'\x00\xe0dqO\x03\x06\x00'), 9: bytearray(b'999535123'), 10: bytearray(b'999884938'), 11: bytearray(b'vivo $2')}",None,[4],None,0,None,None,None,"Row(browser_family=Row(column_size=50044, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound='%E3%82%A6%E3%82%', upper_bound='webprosbot'), device_id=Row(column_size=106306, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound='-100210680', upper_bound='999535123'), device_type=Row(column_size=31286, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound='17MB150WB', upper_bound='vivo $2'), event_date=Row(column_size=4742, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound=datetime.datetime(2021, 1, 12, 0, 0), upper_bound=datetime.datetime(2023, 8, 20, 0, 0)), event_time=Row(column_size=381905, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound=datetime.datetime(2021, 1, 12, 0, 1, 19, 764000), upper_bound=datetime.datetime(2023, 8, 20, 23, 59, 41, 89000)), host=Row(column_size=17186, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound='aashish.techcrea', upper_bound='zachwilson.techd'), os_family=Row(column_size=26501, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound='Android', upper_bound='iOS'), referrer=Row(column_size=69624, value_count=89391, null_value_count=46359, nan_value_count=None, lower_bound='52.20.78.240', upper_bound='zachwilson.tech'), url=Row(column_size=113484, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound='/', upper_bound='/zzageqnf.php?Fp'), user_id=Row(column_size=317883, value_count=89391, null_value_count=1, nan_value_count=None, lower_bound='-1000095488', upper_bound='999884938'))"
0,s3://warehouse/bootcamp/events_unsorted/data/00001-31-6494c81b-af5b-4cb8-9814-851149849b65-0-00001.parquet,PARQUET,1,Row(event_date_year=None),99232,1269939,"{1: 148570, 2: 75427, 3: 50852, 4: 35522, 6: 20849, 7: 426319, 8: 5454, 9: 119968, 10: 346019, 11: 35807}","{1: 99232, 2: 99232, 3: 99232, 4: 99232, 6: 99232, 7: 99232, 8: 99232, 9: 99232, 10: 99232, 11: 99232}","{1: 0, 2: 49299, 3: 0, 4: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 58, 11: 0}",{},"{1: bytearray(b'""/?""""<?=print(93'), 2: bytearray(b'""https://www.goo'), 3: bytearray(b') Bot'), 4: bytearray(b'Android'), 6: bytearray(b'abhishekanand.te'), 7: bytearray(b'(\x83\xb2EX\xb8\x05\x00'), 8: bytearray(b'\x00 \xc9<X\xb8\x05\x00'), 9: bytearray(b'-100210680'), 10: bytearray(b'-1000370060'), 11: bytearray(b'13 Pro Max')}","{1: bytearray(b'/zz.php'), 2: bytearray(b'zachwilson.tech'), 3: bytearray(b'webprosbot'), 4: bytearray(b'iOS'), 6: bytearray(b'zsavi524.techcrf'), 7: bytearray(b'\x88\xb8\x07

In [33]:

start_df = df.repartition(4, col("event_date")).withColumn("event_time", col("event_time").cast("timestamp")) \
    
first_sort_df = start_df.sortWithinPartitions(col("event_date"))

start_df.write.mode("overwrite").saveAsTable("bootcamp.events_unsorted")
first_sort_df.write.mode("overwrite").saveAsTable("bootcamp.events_sorted")

In [34]:
%%sql

SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'sorted' 
FROM demo.bootcamp.events_sorted.files

UNION ALL
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'unsorted' 
FROM demo.bootcamp.events_unsorted.files

size,num_files,sorted
5521392,4,sorted
5556664,4,unsorted


In [35]:
events.write.mode("overwrite").saveAsTable("bootcamp.events")

In [36]:
%%sql
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files FROM demo.bootcamp.events.files;

size,num_files
5515112,4
